In [1]:
# indel_scanner/indel_scanner/main.py

import pysam
import yaml
import os
import argparse
import time
import csv

from multiprocessing import Pool, cpu_count
from functools import partial

In [7]:
from enum import Enum, IntEnum


class Cigar(IntEnum):
	OP_I = pysam.CINS
	OP_D = pysam.CDEL
	OP_M = pysam.CMATCH
	OP_EQ = pysam.CEQUAL
	OP_X = pysam.CDIFF
	OP_N = pysam.CREF_SKIP
	OP_S = pysam.CSOFT_CLIP

In [10]:
import pyfastx

fasta = pyfastx.Fasta("../data/ph/ph_diploid.fa")

In [13]:
fasta['h1tg000003l'].seq

'...'

In [7]:
with pysam.AlignmentFile("/home/peterkad/pkadmaster/data/mutationalscanning_bam/ph/diploid_assembly/ph_plus_unmapped_diploid_v2.bam", "rb") as samfile:
        contigs = samfile.header.references
print(f"found {len(contigs)} contigs.")


found 1409 contigs.


In [11]:
with open("config.yaml", 'r') as f:
        config = yaml.safe_load(f)

In [8]:
# indel_scanner/indel_scanner/scanner.py

import pysam
import pyfastx
import csv
import os

# CIGAR operation codes
CIGAR_OPS = {
    'M': 0, 'I': 1, 'D': 2, 'N': 3, 'S': 4, 'H': 5, 'P': 6, '=': 7, 'X': 8
}

def scan_contig_for_indels(bam_path, fasta_path, contig_name, output_dir, config):
    """
    Scans a single contig in a BAM file for insertions and deletions.

    This function is designed to be run in a separate process.
    """
    # Define the path for this contig's temporary output file
    temp_output_path = os.path.join(output_dir, f"{contig_name}.part.tsv")
    indels_found = []

    # IMPORTANT: File handles must be opened within the worker process
    try:
        samfile = pysam.AlignmentFile(bam_path, "rb")
        fasta = pyfastx.Fasta(fasta_path)
    except Exception as e:
        print(f"Error opening files for contig {contig_name}: {e}")
        return None # Signal failure

    # Iterate over only the reads mapped to the specified contig
    for read in samfile.fetch(contig=contig_name):
        # --- Basic Read Filtering ---
        if (read.is_unmapped or
            read.is_secondary or
            read.is_supplementary or
            read.mapping_quality < config['min_mapping_quality']):
            continue

        ref_pos = read.reference_start
        query_pos = 0

        for op, length in read.cigartuples:
            # --- Insertion Found ---
            if op == CIGAR_OPS['I']:
                if length >= config['min_indel_size']:
                    # Insertion sequence is from the read itself
                    indel_seq = read.query_sequence[query_pos : query_pos + length]
                    indels_found.append([
                        contig_name, ref_pos, 'INS', length, indel_seq,
                        read.query_name, read.mapping_quality
                    ])
                query_pos += length # Consumes query sequence

            # --- Deletion Found ---
            elif op == CIGAR_OPS['D']:
                if length >= config['min_indel_size']:
                    # Deletion sequence is fetched from the reference genome
                    indel_seq = str(fasta[contig_name][ref_pos : ref_pos + length])
                    indels_found.append([
                        contig_name, ref_pos, 'DEL', length, indel_seq,
                        read.query_name, read.mapping_quality
                    ])
                ref_pos += length # Consumes reference sequence

            # --- Operations that consume both query and reference ---
            elif op in [CIGAR_OPS['M'], CIGAR_OPS['='], CIGAR_OPS['X']]:
                ref_pos += length
                query_pos += length

            # --- Operations that consume only the reference ---
            elif op == CIGAR_OPS['N']:
                ref_pos += length

            # --- Operations that consume only the query (soft clipping) ---
            elif op == CIGAR_OPS['S']:
                query_pos += length
    
    samfile.close()

    # --- Write results for this contig to its temporary file ---
    if indels_found:
        with open(temp_output_path, 'w', newline='') as f_out:
            writer = csv.writer(f_out, delimiter='\t')
            writer.writerows(indels_found)
    
    return temp_output_path # Return path for aggregation


In [15]:
scan_contig_for_indels(
	"/home/peterkad/pkadmaster/data/mutationalscanning_bam/ph/diploid_assembly/ph_plus_unmapped_diploid_v2.bam",
	"/home/peterkad/pkadmaster/data/ph/ph_diploid.fa",
	"h1tg000001l",
	"/home/peterkad/pkadmaster/indel_scanner/results/test/indels",
	config)

'/home/peterkad/pkadmaster/indel_scanner/results/test/indels/h1tg000001l.part.tsv'

In [ ]:
from dataclasses import dataclass
from enum import Enum
from abc import ABC

class INDEL_TYPE(Enum):
	INSERTION = "ins"
	DELETION = "del"

@dataclass
class Indel(ABC):
	contig:str
	ref_position:int
	length:int
	prefix_context:str
	suffix_context:str
	read_name:str

@dataclass
class Insertion(Indel):
	inserted_seq:str
	type:INDEL_TYPE = INDEL_TYPE.INSERTION

@dataclass
class Deletion(Indel):
	reference_seq:str
	type:INDEL_TYPE = INDEL_TYPE.DELETION